# Limb Mesenchyme Pathway Heatmap Replication

This notebook replicates the GLI2 pathway enrichment summary for `Limb#mesenchyme#trajectory` and draws a heatmap figure closely matching the provided screenshot style.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

def find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'docs' / 'tutorials').exists() and (p / 'navigo').exists():
            return p
    raise RuntimeError(f'Could not locate unified repo root from {start}')

repo_root = find_repo_root(Path.cwd().resolve())
section_root = repo_root / 'docs' / 'tutorials' / 'resources' / 'knockout' / 'gene_compensation_clean_github'
gli_root = section_root / 'gli'

all_pathway_path = gli_root / 'output' / 'all_pathway_results.csv'
limb_csv_path = gli_root / 'output' / 'hypergeometric_tests' / 'Limb#mesenchyme#trajectory.csv'

all_pathway = pd.read_csv(all_pathway_path)
limb_existing = pd.read_csv(limb_csv_path, index_col=0)

print('Repo root:', repo_root)
print('Section root:', section_root.relative_to(repo_root))
print('Input long table:', all_pathway_path.relative_to(repo_root))
print('Input per-cell matrix:', limb_csv_path.relative_to(repo_root))
print('Rows in long table:', len(all_pathway))

In [ ]:
selected_cell = 'Limb#mesenchyme#trajectory'

pathway_mapping = {
    'HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION': 'EMT',
    'REACTOME_SIGNALING_BY_WNT': 'WNT',
    'REACTOME_SIGNALING_BY_HEDGEHOG': 'Hedgehog',
    'HALLMARK_PI3K_AKT_MTOR_SIGNALING': 'PI3K/Akt/mTOR',
    'HALLMARK_G2M_CHECKPOINT': 'G2M Checkpoint',
    'HALLMARK_TGF_BETA_SIGNALING': 'TGF Beta',
    'HALLMARK_NOTCH_SIGNALING': 'Notch',
    'HALLMARK_APOPTOSIS': 'Apoptosis',
    'HALLMARK_INFLAMMATORY_RESPONSE': 'Inflammatory'
}

pathway_order = [
    'EMT',
    'WNT',
    'Hedgehog',
    'PI3K/Akt/mTOR',
    'G2M Checkpoint',
    'TGF Beta',
    'Notch',
    'Apoptosis',
    'Inflammatory'
]

limb_long = all_pathway[all_pathway['cell_type'] == selected_cell].copy()
limb_long['pathway_short'] = limb_long['pathway'].map(pathway_mapping)

matrix = (
    limb_long.set_index('pathway_short')[['neg_log10_p_pred', 'neg_log10_p_real']]
    .rename(columns={'neg_log10_p_pred': 'Prediction', 'neg_log10_p_real': 'Ground truth'})
    .reindex(pathway_order)
)

pvals = (
    limb_long.set_index('pathway_short')[['p_value_pred', 'p_value_real']]
    .rename(columns={'p_value_pred': 'Prediction', 'p_value_real': 'Ground truth'})
    .reindex(pathway_order)
)

compare = matrix.rename(columns={'Prediction': 'prediction', 'Ground truth': 'groundtruth'})
np.testing.assert_allclose(compare.loc[limb_existing.index].values, limb_existing.values, rtol=0, atol=1e-10)

significant_mask = (pvals < 0.05).any(axis=1)
print('Significant rows (p<0.05 in Prediction or Ground truth):')
print(list(matrix.index[significant_mask]))

matrix.round(2)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_df = matrix.mask(matrix.abs() < 1e-12, 0.0).copy()

fig = plt.figure(figsize=(6.2, 5.0), facecolor='#e5e5e5')
gs = fig.add_gridspec(1, 2, width_ratios=[1.25, 1.0], wspace=0.02)
ax_left = fig.add_subplot(gs[0, 0], facecolor='#e5e5e5')
ax = fig.add_subplot(gs[0, 1], facecolor='#e5e5e5')

sns.heatmap(
    plot_df,
    ax=ax,
    cmap='viridis',
    vmin=0,
    vmax=float(plot_df.to_numpy().max()),
    annot=True,
    fmt='.2f',
    cbar=False,
    linewidths=1.2,
    linecolor='black',
    annot_kws={'fontsize': 11, 'fontweight': 'bold'}
)

ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor', fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)

for t in ax.get_yticklabels()[:3]:
    t.set_fontweight('bold')

for text in ax.texts:
    val = float(text.get_text())
    text.set_color('black' if val >= 3.0 else 'white')

ax_left.set_xlim(0, 1)
ax_left.set_ylim(ax.get_ylim())
ax_left.axis('off')

def draw_bracket(y_top, y_bottom, x=0.28, tick=0.08, lw=1.3):
    ax_left.plot([x, x], [y_top, y_bottom], color='black', lw=lw, clip_on=False)
    ax_left.plot([x, x + tick], [y_top, y_top], color='black', lw=lw, clip_on=False)
    ax_left.plot([x, x + tick], [y_bottom, y_bottom], color='black', lw=lw, clip_on=False)

draw_bracket(0.3, 2.7)
draw_bracket(3.3, 8.5)

ax_left.text(0.08, 1.5, 'Significant\n$p<0.05$', ha='center', va='center', fontsize=14)
ax_left.text(0.09, 5.9, 'Not\nSignificant', ha='center', va='center', fontsize=14)

fig_path = gli_root / 'output' / 'figures' / 'Limb_mesenchyme_pathway_heatmap_replica.png'
fig_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_path, dpi=300, bbox_inches='tight', facecolor=fig.get_facecolor())

print('Saved figure:', fig_path.relative_to(repo_root))
plt.show()